<h1 align="center">Object recognition and computer vision 2026/2027</h1>

<h2 align="center">Assignment 1: Instance-level recognition</h2>

<p align="center"><strong>Adapted from practicals from <a href="https://www.robots.ox.ac.uk/~vgg/practicals/overview/index.html">Andrea Vedaldi and Andrew Zisserman</a><br>
by <a href="https://www.di.ens.fr/~varol/">Gul Varol</a>, <a href="https://www.di.ens.fr/~iroccosp/">Ignacio Rocco</a>, and <a href="https://fcendra.github.io">Fernando Julio Cendra</a>,<br>
and from practicals from <a href="https://cs.nyu.edu/~fouhey/teaching/EECS442_W23">David Fouhey</a> by <a href="https://16lemoing.github.io">Guillaume Le Moing</a></strong></p>

<p align="center">
<img width="800" alt="Example instance-recognition image" src="https://www.di.ens.fr/willow/teaching/recvis17orig/assignment1/images/image12.png"/></p>
<p align="center"></p>

| ✅ **STUDENT ASSIGNMENT NOTEBOOK** |
|:---|
| Complete every cell marked **STUDENT TODO** and every section marked **REQUIRED WRITTEN RESPONSE**. Unless a cell is marked for student work, run it without changing it. |

**STUDENT NAME:** `REPLACE WITH YOUR NAME`  
**STUDENT ID:** `REPLACE WITH YOUR STUDENT ID`  
**EMAIL:** `REPLACE WITH YOUR EMAIL`  

# Guidelines

The purpose of this assignment is that you get hands-on experience with the topics covered in class, which will help you understand these topics better. Therefore, **it is imperative that you do this assignment yourself. No code sharing will be tolerated.**

Once you have completed the assignment, you will submit the `ipynb` file containing **both** code and results. For this, make sure to **run your notebook completely before submitting**.

The `ipynb` must be named using the following format: **A1_LASTNAME_Firstname.ipynb**, and submitted in the **Google Classroom page**.

# How to identify required work

Before submitting, use the notebook search function and search for each of:

- `STUDENT TODO`
- `NotImplementedError`
- `DELETE THIS PLACEHOLDER`

A completed required submission should contain no unfinished occurrence of these
markers in student-response cells.

# Required response tracker

| Task | Required deliverable |
|---|---|
| 1.1 | Code and visualization |
| 1.2 | Code, visualization, and written comment |
| 1.3 | Written response |
| 1.4 | `matching(...)` implementation |
| 1.5 | Ratio-test implementation |
| 1.6 | Written derivation |
| 1.7 | `ransac(...)` implementation |
| 1.8 | Written matrix derivation |
| 1.9 | Written response |
| 1.10 | `compute_homography(...)` implementation |
| 1.11 | `apply_homography(...)` implementation |
| 2.1 | KD-tree word assignment |
| 2.2 | Query BoVW histogram |
| 2.3 | Retrieval scores |
| 2.4 | Optional geometric reranking and analysis |

Use this table as a checklist while progressing through the notebook.

# Goal
The goal of instance-level recognition is to match (recognize) a specific object or scene. Examples include recognizing a specific building, such as Notre Dame, or a specific painting, such as *The Starry Night* by Van Gogh. The object is recognized despite changes in scale, camera viewpoint, illumination conditions, and partial occlusion. An important application is image retrieval: starting from an image of an object of interest (the query), search through an image dataset to retrieve images that contain the target object.

The goal of this assignment is to gain practical experience with methods for specific object recognition. It includes: (i) using SIFT features to obtain sparse matches between two images; (ii) using similarity-covariant detectors to handle changes in viewpoint; (iii) vector-quantizing SIFT descriptors into visual words to enable large-scale retrieval; and (iv) constructing and using an image retrieval system to identify objects.

# Setup environment

This notebook is designed for **Google Colab with a CPU runtime** and also works in a standard Jupyter environment.

Run the dependency cell below once before the imports. It checks the current environment and installs **only packages that are missing**. In a standard Colab runtime, it should normally finish without installing anything. No compiler, runtime restart, or GPU is required.


## Install/check required packages


The original notebook compiled CyVLFeat and downgraded Cython. This version uses OpenCV's maintained SIFT implementation instead.

The following cell checks for NumPy, Matplotlib, SciPy, scikit-image, scikit-learn, Pillow, Requests, and OpenCV. Import names and PyPI package names differ for a few dependencies—for example, `cv2` is provided by `opencv-python-headless`.


In [ ]:
# Install only packages that are missing from the current runtime.
# In a standard Google Colab CPU runtime this should normally be a no-op.

import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "sklearn": "scikit-learn",
    "PIL": "Pillow",
    "requests": "requests",
    "cv2": "opencv-python-headless",
}

missing = [
    package_name
    for import_name, package_name in REQUIREMENTS.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already available.")

print("Dependency check complete; no runtime restart is needed.")


## Imports and compatibility helpers

In [ ]:
# Scientific-Python stack checked by the dependency cell above
from __future__ import annotations

from io import BytesIO
from pathlib import Path
from urllib.parse import urlparse
import platform
import sys
import tarfile
import hashlib
import time

import cv2 as cv
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import requests
from PIL import Image
from scipy.io import loadmat
from skimage.io import imread
from skimage.transform import resize, rotate

# Matplotlib defaults
mpl.rcParams['axes.grid'] = False
mpl.rcParams['figure.dpi'] = 120

IMAGE_CACHE = Path("image_cache")
IMAGE_CACHE.mkdir(exist_ok=True)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "University computer-vision teaching assignment "
        "(educational use; https://fcendra.github.io)"
    )
})


def runtime_report():
    """Print versions that are useful when reproducing a submission."""
    versions = {
        'Python': platform.python_version(),
        'NumPy': np.__version__,
        'OpenCV': cv.__version__,
        'Matplotlib': mpl.__version__,
    }
    print('Runtime ready: all required dependencies are available.')
    print(' | '.join(f'{name} {version}' for name, version in versions.items()))


def _to_gray_uint8(image):
    """Convert a grayscale/RGB image to contiguous uint8 grayscale."""
    array = np.asarray(image)
    if array.ndim == 3:
        if array.shape[2] == 4:
            array = array[..., :3]
        if np.issubdtype(array.dtype, np.floating):
            scale = 255.0 if array.max(initial=0) <= 1.0 else 1.0
            rgb = np.clip(array * scale, 0, 255).astype(np.uint8)
        else:
            rgb = np.clip(array, 0, 255).astype(np.uint8)
        return cv.cvtColor(rgb, cv.COLOR_RGB2GRAY)
    if array.ndim != 2:
        raise ValueError(f'Expected a 2D or 3D image, got shape {array.shape}.')
    if np.issubdtype(array.dtype, np.floating):
        scale = 255.0 if array.max(initial=0) <= 1.0 else 1.0
        array = array * scale
    return np.ascontiguousarray(np.clip(array, 0, 255).astype(np.uint8))


def extract_sift(image, peak_thresh=0.01):
    """OpenCV-backed replacement for ``extract_sift``.

    Returns
    -------
    frames : (N, 4) float32
        Rows are ``(x, y, scale, orientation_in_radians)``.
    descriptors : (N, 128) float32
        Standard SIFT descriptors.

    ``peak_thresh`` is mapped to OpenCV's contrast threshold. The precise
    detector response scale differs from VLFeat, but increasing this value
    has the same intended effect: retaining fewer, higher-contrast features.
    """
    gray = _to_gray_uint8(image)
    threshold = max(float(peak_thresh), 1e-6)
    detector = cv.SIFT_create(contrastThreshold=threshold)
    keypoints, descriptors = detector.detectAndCompute(gray, None)

    if not keypoints:
        return np.empty((0, 4), dtype=np.float32), np.empty((0, 128), dtype=np.float32)

    # OpenCV's KeyPoint.size is a diameter. Dividing by two gives a convenient
    # scale/radius representation; ratios between scales are unchanged.
    frames = np.asarray(
        [[kp.pt[0], kp.pt[1], kp.size / 2.0, np.deg2rad(kp.angle)] for kp in keypoints],
        dtype=np.float32,
    )
    return frames, np.asarray(descriptors, dtype=np.float32)


def plotframes(frames, linewidth=1, ax=None):
    """Plot circular SIFT frames and their dominant orientations."""
    ax = plt.gca() if ax is None else ax
    frames = np.asarray(frames)
    if frames.size == 0:
        return ax
    if frames.ndim != 2 or frames.shape[1] < 4:
        raise ValueError('frames must have shape (N, 4) or wider.')

    x, y, radius, theta = (frames[:, i] for i in range(4))
    radius = np.maximum(radius, 1.0)
    circle_angles = np.linspace(0.0, 2.0 * np.pi, 25)
    circles = [
        np.column_stack((cx + r * np.cos(circle_angles), cy + r * np.sin(circle_angles)))
        for cx, cy, r in zip(x, y, radius)
    ]
    directions = [
        np.asarray([[cx, cy], [cx + r * np.cos(a), cy + r * np.sin(a)]])
        for cx, cy, r, a in zip(x, y, radius, theta)
    ]
    ax.add_collection(LineCollection(circles, linewidths=linewidth, colors='lime'))
    ax.add_collection(LineCollection(directions, linewidths=linewidth, colors='lime'))
    return ax


def pairwise_squared_l2(query, database):
    """Return the exact squared Euclidean distance matrix without a 3D tensor."""
    query = np.asarray(query, dtype=np.float32)
    database = np.asarray(database, dtype=np.float32)
    if query.ndim != 2 or database.ndim != 2 or query.shape[1] != database.shape[1]:
        raise ValueError('query and database must be 2D arrays with equal descriptor dimensions.')
    distances = (
        np.sum(query * query, axis=1, keepdims=True)
        + np.sum(database * database, axis=1)[None, :]
        - 2.0 * query @ database.T
    )
    return np.maximum(distances, 0.0)


def read_rgb_image(path_or_url, max_retries=4):
    path_or_url = str(path_or_url)

    # Handle remote images.
    if path_or_url.startswith(("http://", "https://")):
        parsed = urlparse(path_or_url)
        suffix = Path(parsed.path).suffix or ".jpg"

        cache_name = hashlib.sha256(
            path_or_url.encode("utf-8")
        ).hexdigest() + suffix

        cache_path = IMAGE_CACHE / cache_name

        # Reuse an earlier download.
        if not cache_path.exists():
            for attempt in range(max_retries):
                response = SESSION.get(
                    path_or_url,
                    timeout=30,
                )

                if response.status_code == 429:
                    retry_after = response.headers.get("Retry-After")

                    try:
                        wait_seconds = float(retry_after)
                    except (TypeError, ValueError):
                        wait_seconds = 2 ** attempt

                    print(
                        f"Rate limited; retrying in "
                        f"{wait_seconds:.1f} seconds."
                    )
                    time.sleep(wait_seconds)
                    continue

                response.raise_for_status()
                cache_path.write_bytes(response.content)
                break
            else:
                raise RuntimeError(
                    f"Could not download image after "
                    f"{max_retries} attempts:\n{path_or_url}"
                )

        image_path = cache_path

    # Handle local images.
    else:
        image_path = Path(path_or_url)

    with Image.open(image_path) as image:
        return np.asarray(image.convert("RGB"))


try:
    runtime_report()
except Exception as e:
    print(f"For first-time setup, make sure to first restart the kernel/runtime/session. Error: {e}")


## Download and uncompress data

In [ ]:
from pathlib import Path


def download_file(urls, destination, chunk_size=1024 * 1024):
    """Download once, with HTTPS/HTTP fallback and an atomic final rename."""
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f'Using cached file: {destination}')
        return destination

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.part')
    errors = []
    for url in urls:
        try:
            print(f'Downloading {url}')
            with requests.get(
                url,
                stream=True,
                timeout=(15, 180),
                headers={'User-Agent': 'Mozilla/5.0 (A1 educational notebook)'},
            ) as response:
                response.raise_for_status()
                with temporary.open('wb') as file:
                    for block in response.iter_content(chunk_size=chunk_size):
                        if block:
                            file.write(block)
            temporary.replace(destination)
            print(f'Saved {destination} ({destination.stat().st_size / 1e6:.1f} MB)')
            return destination
        except Exception as error:
            errors.append(f'{url}: {error}')
            temporary.unlink(missing_ok=True)

    raise RuntimeError('All download locations failed:\n' + '\n'.join(errors))


def safe_extract_tar(archive, destination):
    """Extract a tar archive while rejecting path traversal entries."""
    archive = Path(archive)
    destination = Path(destination).resolve()
    with tarfile.open(archive, mode='r:*') as tar:
        for member in tar.getmembers():
            target = (destination / member.name).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe archive member: {member.name}')
        tar.extractall(destination)


DATA_DIR = Path('data')
IMAGE_ARCHIVE = Path('A1_images.tar.gz')
IMAGE_URLS = [
    'https://www.di.ens.fr/willow/teaching/recvis_orig/assignment1/A1_images.tar.gz',
    'http://www.di.ens.fr/willow/teaching/recvis_orig/assignment1/A1_images.tar.gz',
]

if not DATA_DIR.exists():
    download_file(IMAGE_URLS, IMAGE_ARCHIVE)
    print('Extracting assignment images...')
    safe_extract_tar(IMAGE_ARCHIVE, Path('.'))
    print('Extraction complete.')
else:
    print(f'Using existing data directory: {DATA_DIR.resolve()}')


# Part 1: Sparse features for matching specific objects in images

## Feature point detection

The SIFT feature has both a *detector* and a *descriptor*. The *detector* used in SIFT corresponds to the "difference of Gaussians" (DoG) detector, which is an approximation of the "Laplacian of Gaussian"  (LoG) detector.

We will start by computing and visualizing the SIFT feature detections (usually called frames) for two images of the same object (a building facade). Load an image, rotate and scale it, and then display the original and transformed pair:

In [ ]:
# Load an image
im1 = imread('data/oxbuild_lite/all_souls_000002.jpg')

# Let the second image be a rotated and scaled version of the first
padded = np.pad(im1, ((0, 0), (200, 200), (0, 0)), mode='constant')
im1prime = rotate(padded, angle=35, resize=True, order=1, preserve_range=True)
h, w, _ = im1prime.shape
im1prime = resize(
    im1prime,
    (int(0.7 * h), int(0.7 * w)),
    order=1,
    preserve_range=True,
    anti_aliasing=True,
).astype(np.uint8)

f, (ax1, ax2) = plt.subplots(1, 2)
ax1.imshow(im1)
ax2.imshow(im1prime)
plt.show()

A SIFT frame is a circle with an orientation and is specified by four parameters: the center $x$, $y$, the scale $s$, and the rotation $\theta$ (in radians), resulting in a vector of four parameters $(x, y, s, \theta)$.

Now compute and visualise the SIFT feature detections (frames):

In [ ]:
def rgb2gray(rgb):
    """Convert RGB uint8/float images to float32 grayscale in [0, 1]."""
    gray = _to_gray_uint8(rgb)
    return gray.astype(np.float32) / 255.0

In [ ]:
# Compute SIFT features for each

[frames1, descrs1] = extract_sift(rgb2gray(im1), peak_thresh=0.01)

[frames1prime, descrs1prime] = extract_sift(rgb2gray(im1prime), peak_thresh=0.01)

f, (ax1, ax2) = plt.subplots(1, 2)
plt.sca(ax1)
plt.imshow(im1)
plotframes(frames1, linewidth=1)

plt.sca(ax2)
plt.imshow(im1prime)
plotframes(frames1prime, linewidth=1)

plt.show()

Examine the second image and its rotated and scaled version and convince yourself that the detections overlap the same scene regions (even though the circles have moved and their radii have changed).
This demonstrates that the detection process is covariant (or equivariant) with translations, rotations, and isotropic scaling. This class of transformations is known as the similarity group.

### :: TASK 1.1 ::

Now repeat the exercise with a pair of natural images.

Start by loading the second one: `data/oxbuild_lite/all_souls_000015.jpg`

Plot the images and feature frames. Again you should see that many of the detections overlap the same scene region. Note that, while repeatability occurs for the pair of natural views, it is much better for the synthetically rotated pair.

| 🟧 **REQUIRED CODE — TASK 1.1** |
|:---|
| Complete the code cell immediately below. It must define `im2`, `frames2`, and `descrs2`, and display both natural images with their SIFT frames. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.1 (REQUIRED)
# Expected outputs: im2, frames2, descrs2
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
# Load the second natural image.
im2 = imread('data/oxbuild_lite/all_souls_000015.jpg')

# TODO:


raise NotImplementedError("Complete Task 1.1 and remove this line.")

The number of detected features can be controlled by changing the peakThreshold option. A larger value will select features that correspond to higher contrast structures in the image.

### :: TASK 1.2 ::

***For the same image, produce 3 sub-figures with different values of peakThreshold. Comment.***

| 🟧 **REQUIRED CODE + WRITTEN COMMENT — TASK 1.2** |
|:---|
| Produce three subfigures using different `peak_thresh` values, then explain the observed change in the written-response cell. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.2 (REQUIRED)
# Expected outputs: a figure with three subplots
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
# You may use these values or choose three clearly different thresholds.
peak_thresholds = [0.005, 0.02, 0.08]

# TODO:


raise NotImplementedError("Complete the Task 1.2 visualization and remove this line.")

| 🟧 **REQUIRED WRITTEN RESPONSE — TASK 1.2** |
|:---|
| Write approximately 3–6 sentences explaining how increasing the peak threshold affects the number, location, stability, and computational cost of detected features. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER HERE.*

## Feature point description and matching

### Introduction to feature descriptors

The parameters $(t_x, t_y, s, \theta)$ of the detected *frames*,  can be used to extract a **scaled and oriented RGB patch** around $(t_x,t_y)$, used to *describe* the feature point.

The simplest possible descriptor would be to (i) resize these patches to a common size (e.g., 30×30) and (ii) flatten each patch into a vector. In practice, however, we use more sophisticated descriptors such as SIFT, which is based on histograms of gradient orientations.

### :: TASK 1.3 ::


***What is the interest of using SIFT descriptors over these flattened RGB patches?***


| 🟧 **REQUIRED WRITTEN RESPONSE — TASK 1.3** |
|:---|
| Write approximately 3–5 sentences comparing SIFT descriptors with raw, flattened RGB patches. Address invariance and robustness. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER HERE.*

### Descriptor matching
SIFT descriptors are 128-dimensional vectors, and can be directly *matched* to find correspondences between images. We will start with the simplest matching scheme (first nearest neighbour of descriptors in terms of Euclidean distance) and then add more sophisticated methods to eliminate any mismatches.

### :: TASK 1.4 ::


For each descriptor in im1, assign a matching descriptor in im2 by picking its first nearest neighbour.

Populate the second column of the matches vector.

| 🟧 **REQUIRED FUNCTION — TASK 1.4** |
|:---|
| Implement `matching(frames1, frames2, descrs1, descrs2)`. Do not rename the function or change its arguments. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.4 (REQUIRED)
# Expected outputs: matches with shape (len(frames1), 2)
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
def matching(frames1, frames2, descrs1, descrs2):
    # Number of detections in image 1.
    N_frames1 = frames1.shape[0]

    # Each row must be [index_in_image_1, index_in_image_2].
    matches = np.zeros((N_frames1, 2), dtype=np.int64)
    matches[:, 0] = np.arange(N_frames1)

    if N_frames1 == 0:
        return matches
    if descrs2.shape[0] == 0:
        raise ValueError('Image 2 has no descriptors to match against.')

    # TODO:
    # Compute Euclidean descriptor distances and place the first-nearest-
    # neighbour index from image 2 into matches[:, 1].
    #
    # The provided pairwise_squared_l2(...) helper may be used.

    raise NotImplementedError("Complete Task 1.4 and remove this line.")

    return matches

Compute the matches.

In [ ]:
matches = matching(frames1, frames2, descrs1, descrs2)

Plot the matches.

In [ ]:
from matplotlib.collections import LineCollection


def plot_matches(im1, im2, frames1, frames2, matches):
    def to_rgb_float(image):
        image = np.asarray(image)

        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2)

        image = image[..., :3].astype(np.float32)

        if image.size > 0 and image.max() > 1.0:
            image /= 255.0

        return np.clip(image, 0.0, 1.0)

    im1 = to_rgb_float(im1)
    im2 = to_rgb_float(im2)

    frames1 = np.asarray(frames1)
    frames2 = np.asarray(frames2)
    matches = np.asarray(matches, dtype=np.int64)

    # Display at most 4000 matches.
    # This should be enough to visualize the overall matching quality
    # without cluttering the figure.
    if len(matches) > 4000:
        selected = np.linspace(
            0,
            len(matches) - 1,
            4000,
            dtype=np.int64,
        )
        matches = matches[selected]

    height = max(im1.shape[0], im2.shape[0])
    width = im1.shape[1] + im2.shape[1]

    canvas = np.zeros((height, width, 3), dtype=np.float32)

    canvas[:im1.shape[0], :im1.shape[1]] = im1
    canvas[
        :im2.shape[0],
        im1.shape[1]:im1.shape[1] + im2.shape[1],
    ] = im2

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(canvas)

    if len(matches) > 0:
        indices1 = matches[:, 0]
        indices2 = matches[:, 1]

        points1 = frames1[indices1, :2].astype(np.float32)
        points2 = frames2[indices2, :2].astype(np.float32).copy()

        # Shift coordinates of features in the second image.
        points2[:, 0] += im1.shape[1]

        segments = np.stack((points1, points2), axis=1)

        # Reproduce Matplotlib's automatic multicolor line cycle.
        color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        line_colors = [
            color_cycle[index % len(color_cycle)]
            for index in range(len(segments))
        ]

        line_collection = LineCollection(
            segments,
            colors=line_colors,
            linewidths=0.5,
            alpha=0.8,
        )
        ax.add_collection(line_collection)

        # Keep feature-point markers green, as in the original function.
        all_points = np.concatenate((points1, points2), axis=0)

        ax.scatter(
            all_points[:, 0],
            all_points[:, 1],
            s=5,
            c="green",
        )

    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.axis("off")

    plt.tight_layout(pad=0)
    plt.show()


plot_matches(im1, im2, frames1, frames2, matches)

## Improving SIFT matching (i) using Lowe’s second nearest neighbour test


Lowe introduced a second nearest neighbour (2nd NN) test to identify, and hence remove, ambiguous matches. The idea is to identify distinctive matches by a threshold on the ratio of first to second NN distances.

The ratio is:  $$NN_{ratio} = \frac{1^{st}\text{NN distance}}{2^{nd}\text{NN distance}}.$$



### :: TASK 1.5 ::

***For each descriptor in im1, compute the ratio between the first and second nearest neighbour distances.***

***Populate the ratio vector.***

| 🟧 **REQUIRED CODE — TASK 1.5** |
|:---|
| Compute the first/second-nearest-neighbour ratio for every descriptor in image 1. Preserve the variables `ratio` and `matches`. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.5 (REQUIRED)
# Expected outputs: ratio and matches
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
# Allocate the required arrays.
ratio = np.zeros(frames1.shape[0], dtype=np.float32)
matches = np.zeros((frames1.shape[0], 2), dtype=np.int64)

descrs1_float = descrs1.astype(np.float32)
descrs2_float = descrs2.astype(np.float32)

# TODO:
# For every descriptor in image 1:
#   1. Compute its distance to every descriptor in image 2.
#   2. Find the first and second nearest neighbours.
#   3. Store [image_1_index, first_nn_image_2_index] in matches.
#   4. Store first_nn_distance / second_nn_distance in ratio.
# Handle a zero second-neighbour distance safely.

raise NotImplementedError("Complete Task 1.5 and remove this line.")

The ratio vector will now be used to retain only the matches that are below a given threshold.

A value of $NN_{threshold} = 0.8$ is often a good compromise between losing too many matches and rejecting mismatches.

In [ ]:
NN_threshold = 0.8

filtered_indices = np.flatnonzero(ratio < NN_threshold)
filtered_matches = matches[filtered_indices, :]

Plot the filtered matches.

In [ ]:
plot_matches(im1, im2, frames1, frames2, filtered_matches)

## Improving SIFT matching (ii) by geometric verification

In addition to the 2nd NN test, we can also require consistency between the matches and a geometric transformation between the images. For the moment we will look for matches that are consistent with a similarity transformation:

$$\begin{bmatrix} x_2 \\ y_2 \end{bmatrix} =
sR(\theta) \begin{bmatrix} x_1 \\ y_1 \end{bmatrix} + \begin{bmatrix} t_x \\ t_y \end{bmatrix} $$

which consists of a rotation by $\theta$, an isotropic scaling (i.e. same in all directions) by s, and a translation by a vector $(t_x, t_y)$. This transformation is specified by four parameters $(s,\theta,t_x,t_y)$ and can be computed from a single correspondence between SIFT detections in each image.

### :: TASK 1.6 ::

Given a detected feature with parameters $(x_1, y_1, s_1, \theta_1)$ in image $1$ matching a feature $(x_2, y_2, s_2, \theta_2)$ in image $2$, work out how to find out the parameters $(t_x,t_y,s,\theta)$ of the transformation mapping points from image $1$ to image $2$.

| 🟧 **REQUIRED WRITTEN RESPONSE — TASK 1.6** |
|:---|
| Provide the equations for $\theta$, $s$, $t_x$, and $t_y$. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER BELOW.*

$$\theta=\text{COMPLETE HERE}$$
$$s=\text{COMPLETE HERE}$$
$$\begin{bmatrix} t_x \\ t_y \end{bmatrix} =\text{COMPLETE HERE}$$

The matches consistent with a similarity can then be found using the RANSAC algorithm, described by the following steps:

For each sampled tentative correspondence:

* compute the similarity transformation;
* map all the SIFT detections in one image to the other using this transformation;
* accept matches that are within a threshold distance to the mapped detection (inliers);
* count the number of accepted matches;
* optionally, fit a more accurate affine transformation or homography to the accepted matches and re-validate them.

Finally, choose the transformation with the highest count of inliers.

After this algorithm the inliers are consistent with the transformation and are retained, and most mismatches should now be removed.

### :: TASK 1.7 ::

| 🟧 **REQUIRED FUNCTION — TASK 1.7** |
|:---|
| Complete the transformation-estimation section of `ransac(frames1, frames2, matches, N_iters=500, dist_thresh=15)`. Keep the signature unchanged. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.7 (REQUIRED)
# Expected outputs: tnf and best_inliers_indices
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
def ransac(frames1, frames2, matches, N_iters=500, dist_thresh=15):
    # Initialize.
    max_inliers = 0
    tnf = None
    best_inliers_indices = np.empty(0, dtype=np.int64)

    if len(matches) == 0:
        return tnf, best_inliers_indices

    # Run random sampling.
    for it in range(N_iters):
        # Pick a random tentative correspondence.
        match_index = np.random.randint(0, matches.shape[0])
        i = matches[match_index, 0]
        j = matches[match_index, 1]
        x_1, y_1, s_1, theta_1 = frames1[i, :]
        x_2, y_2, s_2, theta_2 = frames2[j, :]

        # TODO: estimate the similarity transformation.
        # theta =
        # s =
        # t_x =
        # t_y =

        raise NotImplementedError(
            "Complete the transformation estimation in Task 1.7 "
            "and remove this line."
        )

        # Evaluate the estimated transformation.
        X_1 = frames1[matches[:, 0], 0]
        Y_1 = frames1[matches[:, 0], 1]
        X_2 = frames2[matches[:, 1], 0]
        Y_2 = frames2[matches[:, 1], 1]

        X_1_prime = s * (
            X_1 * np.cos(theta) - Y_1 * np.sin(theta)
        ) + t_x
        Y_1_prime = s * (
            X_1 * np.sin(theta) + Y_1 * np.cos(theta)
        ) + t_y

        dist = np.sqrt(
            (X_1_prime - X_2) ** 2 +
            (Y_1_prime - Y_2) ** 2
        )

        inliers_indices = np.flatnonzero(dist < dist_thresh)
        num_of_inliers = len(inliers_indices)

        # Keep the best hypothesis.
        if num_of_inliers > max_inliers:
            max_inliers = num_of_inliers
            best_inliers_indices = inliers_indices
            tnf = [t_x, t_y, s, theta]

    return tnf, best_inliers_indices

In [ ]:
tnf, inliers_indices = ransac(frames1, frames2, matches)
filtered_matches = matches[inliers_indices, :]

Plot matches filtered with RANSAC.

In [ ]:
plot_matches(im1, im2, frames1, frames2, filtered_matches)

## Augmented reality via template matching

<p align="center"><img height="200" alt="Augmented-reality example" src="https://www.di.ens.fr/willow/teaching/recvis_orig/assignment1/augmented_reality.png"/></p>

Now that we are able to draw consistent matches between pairs of images, we can use them to start replacing things in our reality. Imagine that we have a template
image and this template appears in the world but viewed at a different angle. We can fit a homography mapping between the template and the scene. Once we know the homography, we can use any new image to replace the template in the scene.

In [ ]:
# Load the template and the scene
template = imread('data/augmented_reality/template.png')
scene = imread('data/augmented_reality/scene.png')
h, w, _ = scene.shape
factor = h / template.shape[0]
template_width = int(round(template.shape[1] * factor))
templatexl = resize(
    template,
    (h, template_width),
    preserve_range=True,
    anti_aliasing=True,
).astype(np.uint8)

f, (ax1, ax2) = plt.subplots(1, 2)

ax1.imshow(scene)
ax2.imshow(template)
ax1.title.set_text('Scene')
ax2.title.set_text('Template')
ax1.axis('off')
ax2.axis('off')
plt.show()

In [ ]:
# Compute matches between the template and the scene

[frames_scene, descrs_scene] = extract_sift(rgb2gray(scene), peak_thresh=0.01)
[frames_template, descrs_template] = extract_sift(rgb2gray(template), peak_thresh=0.01)

matches = matching(frames_scene, frames_template, descrs_scene, descrs_template)

plot_matches(scene, templatexl, frames_scene, factor * frames_template, matches)

In [ ]:
# Filter these matches with RANSAC

tnf, inliers_indices = ransac(frames_scene, frames_template, matches)
filtered_matches = matches[inliers_indices, :]

plot_matches(scene, templatexl, frames_scene, factor * frames_template, filtered_matches)

We wish to fit a homography using the filtered matches $i$ in $1 .. N$, that is, to find the transformation $H$ which verifies for each point $(x_1^i, y_1^i)$ from the "scene" and its correspondence $(x_2^i, y_2^i)$ in the "template":

$$ \lambda \mathbf{x_2^i} = H \mathbf{x_1^i} $$

with points expressed in homogeneous coordinates:
$$H=\begin{bmatrix} h_{11} & h_{12} & h_{13} \\ h_{21} & h_{22} & h_{23} \\ h_{31} & h_{32} & h_{33} \end{bmatrix}, \,\,\,
\mathbf{x_1^i}=\begin{bmatrix} x_1^i \\ y_1^i \\ 1 \end{bmatrix}, \,\,\,
\mathbf{x_2^i}=\begin{bmatrix} x_2^i \\ y_2^i \\ 1 \end{bmatrix}.
$$

The scale ($\lambda$) is arbitrary. We can also write this as:
$$\mathbf{x_2^i}\times (H \mathbf{x_1^i}) = 0,$$
with $\times$ denoting the vector cross product.

### :: TASK 1.8 ::

Given a match $(\mathbf{x_1^i}, \mathbf{x_2^i})$ and a flattened view of the matrix $H$:
$$\mathbf{\bar{h}}= \begin{bmatrix} h_{11} & h_{12} & h_{13} & h_{21} & h_{22} & h_{23} & h_{31} & h_{32} & h_{33} \end{bmatrix}^T,$$

find a matrix $A_i$ such that the previous equation is expressed as a system

$$A_i \mathbf{\bar{h}} = 0$$

of linearly independent equations.

| 🟧 **REQUIRED WRITTEN RESPONSE — TASK 1.8** |
|:---|
| Write the explicit $2\times9$ matrix $A_i$ for one correspondence. Equivalent sign conventions are acceptable when they define the same homogeneous system. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER BELOW.*

$$A_i=\text{COMPLETE HERE}$$

### :: TASK 1.9 ::

***How many linearly independent equations each match gives us? How many degrees of freedom does $H$ have? Deduce the minimal number of matches to find a solution.***

| 🟧 **REQUIRED WRITTEN RESPONSE — TASK 1.9** |
|:---|
| Answer all three questions. Include the non-degeneracy condition required for the minimal correspondence set. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER HERE.*

The complete system $A \mathbf{\bar{h}} = 0$ is built by concatenating the rows of $A_i$ for every match $i$ in $1 .. N$.

A non-zero solution $\mathbf{\bar{h}}$ to this homogeneous least squares system is obtained by the singular value decomposition (SVD) of $A$. It is the singular vector corresponding to the smallest singular value. See for example [here](https://en.wikipedia.org/wiki/Singular_value_decomposition#Solving_homogeneous_linear_equations) if you are curious about the maths behind it.

### :: TASK 1.10 ::

| 🟧 **REQUIRED FUNCTION — TASK 1.10** |
|:---|
| Complete `compute_homography(frames1, frames2, matches)` without changing its signature. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.10 (REQUIRED)
# Expected outputs: a 3×3 homography H
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
def compute_homography(frames1, frames2, matches):
    x_1 = frames1[matches[:, 0], 0]
    y_1 = frames1[matches[:, 0], 1]
    x_2 = frames2[matches[:, 1], 0]
    y_2 = frames2[matches[:, 1], 1]

    # TODO:
    # HINT: using the equations derived in Task 1.8.
    # A = ...

    raise NotImplementedError("Construct A for Task 1.10 and remove this line.")

    A = np.asarray(A, dtype=np.float64)

    # Solve the homogeneous least-squares system using SVD.
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1, :].reshape(3, 3)

    # Normalize the arbitrary homogeneous scale.
    H /= H[2, 2]

    return H

In [ ]:
H = compute_homography(frames_scene, frames_template, filtered_matches)
print(H)

Now we will use the homography to replace the template in the scene by a new image.

### :: TASK 1.11 ::

Complete the function below which takes as input the homography $H$, a template and the scene and which returns an image which is the result of warping the template into the scene and overlaying it on top of the original scene using alpha compositing.

Use bilinear interpolation to determine pixel colors in the warped template given those in the original template. This deliberately uses nested Python loops for clarity; the final cell may take roughly 30–60 seconds on a Colab CPU runtime.

<p align="center"><img height="200" alt="Bilinear-interpolation weights" src="https://gabrielgambetta.com/computer-graphics-from-scratch/images/bilinear-texture-weights.png"/></p>

More precisely, given a pixel location in the scene, let $C=(c_x,c_y)$ be its projection onto the template by applying the homography $H$. If $C$ has four valid neighbours in the template, the resulting color is a weighted sum using $(1-f_x)(1-f_y)$, $f_x(1-f_y)$, $(1-f_x)f_y$, and $f_xf_y$ for the top-left (TL), top-right (TR), bottom-left (BL), and bottom-right (BR) pixels, respectively. This means checking $0 \le c_x < W-1$ and $0 \le c_y < H_t-1$, where $W$ and $H_t$ are the template width and height.

| 🟧 **REQUIRED FUNCTION — TASK 1.11** |
|:---|
| Complete the bilinear-interpolation and mask-update section of `apply_homography(H, template, scene)`. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 1.11 (REQUIRED)
# Expected outputs: the composited RGB image result
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
def apply_homography(H, template, scene):
    height, width, _ = scene.shape

    # Warped template and visibility mask.
    w_template = np.zeros((height, width, 3), dtype=np.uint8)
    mask = np.zeros((height, width, 1), dtype=np.uint8)

    # For each point in the scene:
    for y in range(height):
        for x in range(width):
            # Map the scene pixel into template coordinates.
            c = np.dot(H, [x, y, 1])
            c /= c[2]
            c = c[:2]

            # TODO:
            # Now check if the projected point ("C") is within the bounds of the template image
            # and update the mask ("mask[y, x]") accordingly.

            # If it is inside the bounds, update the color of the warped template ("w_template[y, x]")
            # using colors from neighbouring pixels in the original "template" image (bilinear interpolation).
            # Note that "w_template" expects np.uint8 colors (discrete RGB values in 0..255)

            raise NotImplementedError(
                "Complete the inner interpolation block for Task 1.11 "
                "and remove this line."
            )

    # Alpha compositing.
    result = mask * w_template + (1 - mask) * scene
    return result

Choose an image to replace the template in the scene.

In [ ]:
name = "ens"  # options ["willow", "monk", "ens"]

In [ ]:
# Replace the template in the scene

f, (ax1, ax2, ax3) = plt.subplots(1, 3)
replacement = imread(f'data/augmented_reality/{name}.png')
replacement = (resize(replacement, (template.shape[0], template.shape[1])) * 255).astype(np.uint8)
result = apply_homography(H, replacement, scene)

ax1.imshow(template)
ax2.imshow(replacement)
ax3.imshow(result)
ax1.title.set_text('Template')
ax2.title.set_text('Replacement')
ax3.title.set_text('Result')
ax1.axis('off')
ax2.axis('off')
ax3.axis('off')
plt.show()

# Part 2: Compact descriptors for image retrieval

In large-scale retrieval, the goal is to match a query image to a large database of images (for example, the web or Wikipedia).

The quality of an image match is measured as the number of geometrically verified feature correspondences between the query and a database image. While the techniques discussed in Part 1 are sufficient to do this, in practice they require too much memory to store the SIFT descriptors for all the detections in all the database images.

In this part we will see how we can compute a *global* image descriptor from the set of SIFT descriptors using the bag-of-visual-words (BoVW) approach.

Then, we will see how these global descriptors can be used to rapidly retrieve a shortlist of candidate database images given a query image. Finally, we will see how to re-rank the shortlist of candidates using a geometric verification technique that requires only the *detector frames* and their assigned visual word indices; remember the SIFT descriptors are only used to compute the compact BoVW descriptors and then discarded.


## Download preprocessed dataset of paintings

The dataset is approximately **168 MB**. Its first download may take about a minute in Colab; later runs reuse the cached file.

In [ ]:
PAINTING_DATA = Path('paintings_imdb_SIFT_10k_preprocessed.mat')
PAINTING_URLS = [
    'https://www.di.ens.fr/willow/teaching/recvis_orig/assignment1/paintings_imdb_SIFT_10k_preprocessed.mat',
    'http://www.di.ens.fr/willow/teaching/recvis_orig/assignment1/paintings_imdb_SIFT_10k_preprocessed.mat',
]

download_file(PAINTING_URLS, PAINTING_DATA)


## Load preprocessed dataset of paintings

We will now load the preprocessed dataset of paintings. The construction of this dataset has involved several steps.


1. SIFT features were extracted from every painting in the dataset.
2.   A global vocabulary of SIFT descriptors was computed using K-means clustering. These are the visual words of our dataset.
3.  The SIFT features for each painting were assigned to the nearest word, and a compact descriptor was generated for each painting. This compact descriptor consists in the normalized histogram of words. The histogram normalization itself involves 3 different steps:
    - a) TF-IDF weighting: each word is re-weighted according to its TF-IDF value. This removes weights to words that are very common and therefore not too descriptive.
    - b) Square-rooting: each element is square-rooted
    - c) L2-normalization: The whole histogram is L2-normalized.
    




In [ ]:
imdb=loadmat(PAINTING_DATA)

feature_vocab = np.transpose(imdb['vocab'])
imdb_hists = imdb['index']
imdb_tfidf = imdb['idf']

imdb_url = lambda idx: imdb['images'][0][0][3][0][idx].item()

num_words = feature_vocab.shape[0]
num_paintings = imdb_hists.shape[1]

print('The vocabulary of SIFT features contains %s  visual words' % num_words)
print('The dictionary index contains %s histograms corresponding to each painting in the dataset' % num_paintings)
print('The TF-IDF vector has shape ' + str(imdb_tfidf.shape))

In [ ]:
painting = imread('data/queries/mistery-painting1.jpg')

[frames, descrs] = extract_sift(rgb2gray(painting), peak_thresh=0.01)

plt.figure()
plt.imshow(painting)
plotframes(frames,linewidth=1)

### :: TASK 2.1 ::

Construct a KDTree of the vocabulary for fast NN search. Then use the KDTree to find the closest word in the vocabulary to each descriptor of the query image.

| 🟧 **REQUIRED CODE — TASK 2.1** |
|:---|
| Construct the vocabulary KD-tree and define `vocabulary_tree`, `word_distances`, `word_indices`, and `query_words`. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 2.1 (REQUIRED)
# Expected outputs: vocabulary_tree, word_distances, word_indices, query_words
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
from sklearn.neighbors import KDTree

# TODO:
# 1. Construct a KDTree from feature_vocab.
# 2. Query the nearest visual word for every query descriptor.
# 3. Store the one-dimensional integer word assignments in query_words.

raise NotImplementedError("Complete Task 2.1 and remove this line.")

### :: TASK 2.2 ::
Compute the compact BoVW descriptor of the query image.

| 🟧 **REQUIRED CODE — TASK 2.2** |
|:---|
| Construct the raw visual-word histogram `query_hist`. The following provided cell performs TF–IDF and normalization. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 2.2 (REQUIRED)
# Expected outputs: query_hist with shape (num_words, 1)
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
# TODO:
# Count the occurrences of every visual word in query_words.
# Include zero-count vocabulary entries and reshape the result to (num_words, 1).

raise NotImplementedError("Complete Task 2.2 and remove this line.")

In [ ]:
# Process histogram: TF-IDF, Hellinger (square-root), then L2 normalization.
query_hist = query_hist * imdb_tfidf
query_hist = np.sqrt(np.maximum(query_hist, 0.0))
query_norm = np.linalg.norm(query_hist)
if query_norm <= np.finfo(float).eps:
    raise ValueError('The query produced an empty/zero visual-word histogram.')
query_hist = query_hist / query_norm

### :: TASK 2.3 ::

Compute the matching score with each image from the database.

| 🟧 **REQUIRED CODE — TASK 2.3** |
|:---|
| Compute `scores`, containing one similarity score for every database painting. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 2.3 (REQUIRED)
# Expected outputs: scores with one value per database painting
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
# TODO:
# Compute the similarity between the normalized query histogram and every
# normalized database histogram.
scores = ...

raise NotImplementedError("Complete Task 2.3 and remove this line.")

In [ ]:
# sort in descending order
scores_sorted_idx = np.argsort(-scores)
scores_sorted = scores.ravel()[scores_sorted_idx]

In [ ]:
# Plot the top retrieval results.
N = 10
top_N_idx = scores_sorted_idx.ravel()[:N]
top_N_scores = scores_sorted.ravel()[:N]

fig, axes = plt.subplots(
    2,
    5,
    figsize=(15, 7),
)

for rank, ax in enumerate(axes.ravel()):
    image_index = int(top_N_idx[rank])
    url = imdb_url(image_index)

    try:
        image = read_rgb_image(url)

        ax.imshow(image)
        ax.set_title(
            f"Rank {rank + 1}\n"
            f"score {top_N_scores[rank]:.2f}"
        )

    except Exception as error:
        ax.text(
            0.5,
            0.5,
            "Image unavailable",
            ha="center",
            va="center",
        )
        ax.set_title(f"Rank {rank + 1}")
        print(f"Failed to load {url}: {error}")

    ax.axis("off")

    # Avoid sending requests in a rapid burst.
    time.sleep(0.25)

plt.tight_layout()
plt.show()

### :: TASK 2.4 (optional) ::

Using the frame information and the assigned words of each descriptor, extract matching frames between the query and the shortlist of database images and perform geometric verification using a similarity transformation with no rotation.

Compute the inlier scores and use it to re-rank this short list. Analyze the gap in the inlier count of the correct match and the incorrect ones.

| 🟦 **OPTIONAL CODE + ANALYSIS — TASK 2.4** |
|:---|
| If you attempt this task, set `RUN_OPTIONAL_TASK_2_4 = True`, implement geometric reranking, and discuss the inlier-count gap in the optional response cell. Leave the flag as `False` to skip this cell cleanly during a full notebook run. |

In [ ]:
# ============================================================
# STUDENT TODO — TASK 2.4 (OPTIONAL)
# Expected outputs: reranked_indices and inlier scores for the shortlist
# Complete the indicated section and remove the
# NotImplementedError before continuing.
# ============================================================
RUN_OPTIONAL_TASK_2_4 = False

imdb_frames = lambda idx: imdb['images'][0][0][5][0][idx]
imdb_words = lambda idx: imdb['images'][0][0][6][0][idx]


def standardize_frames(raw_frames):
    """Return database frames as an N×4 floating-point array."""
    # TODO
    raise NotImplementedError("Complete standardize_frames for optional Task 2.4.")


def standardize_words(raw_words, vocabulary_size):
    """Return zero-based visual-word indices."""
    # TODO
    raise NotImplementedError("Complete standardize_words for optional Task 2.4.")


def word_matches(words1, words2, max_pairs_per_word=100):
    """Return tentative index pairs sharing the same visual word."""
    # TODO
    raise NotImplementedError("Complete word_matches for optional Task 2.4.")


def ransac_scale_translation(
    frames1,
    frames2,
    matches,
    N_iters=300,
    dist_thresh=15,
    seed=0,
):
    """Return inlier indices for a scale-and-translation model."""
    # TODO
    raise NotImplementedError(
        "Complete ransac_scale_translation for optional Task 2.4."
    )


# TODO:
# 1. Take a shortlist from scores_sorted_idx.
# 2. Build word-based tentative matches for each candidate.
# 3. Compute a geometric inlier score.
# 4. Re-rank the shortlist by decreasing inlier count.

if RUN_OPTIONAL_TASK_2_4:
    raise NotImplementedError("Complete the optional Task 2.4 pipeline.")
else:
    print("Optional Task 2.4 skipped. Set RUN_OPTIONAL_TASK_2_4 = True to attempt it.")

| 🟦 **OPTIONAL WRITTEN RESPONSE — TASK 2.4** |
|:---|
| Discuss whether the correct instance has a clear inlier-count gap over incorrect candidates and explain why shared visual words alone can create false tentative matches. **Replace the placeholder below with your response; do not delete the heading.** |

### Your response

*DELETE THIS PLACEHOLDER AND WRITE YOUR ANSWER HERE.*

# Final submission checklist

Before exporting or submitting the notebook:

- ☐ Fill in your name, student ID, and email at the top.
- ☐ Complete Tasks 1.1–1.11 and Tasks 2.1–2.3.
- ☐ Replace every required written-response placeholder.
- ☐ Search for `NotImplementedError` and remove every required occurrence.
- ☐ Search for `DELETE THIS PLACEHOLDER` and replace every required occurrence.
- ☐ Run the notebook from top to bottom in a fresh kernel/runtime.
- ☐ Confirm that all required figures and numerical outputs are visible.
- ☐ Confirm that no cell reports an exception.
- ☐ Save the executed `.ipynb` using the required filename format.

Task 2.4 is optional; unfinished markers inside that optional section may remain
only when you are not submitting the optional task.

## AUTHORSHIP STATEMENT

I declare that the preceding work was the sole result of my own effort and that
I have not used any code or results from third parties except where explicitly
permitted and acknowledged under the course rules.

**NAME:** `REPLACE WITH YOUR NAME`